# Particles → Lasers → Voltage → Schrödinger Equation → Engineering Value

This notebook connects:
- elementary particles and antimatter vocabulary,
- photon energy and laser light,
- photodetector current and voltage,
- ADC quantization,
- the 1-D Schrödinger equation as an eigenvalue problem,
- and realistic ways this physics becomes engineering value.

The practical question is not "How do I sell the Schrödinger equation?"

It is:

> What useful engineering capability can I build because I can model quantum systems numerically?


In [ ]:
import sympy as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sp.init_printing()


## 1. Particle table

We start with a small teaching table.


In [ ]:
particles = pd.DataFrame({
    "particle": ["electron", "positron", "photon"],
    "charge_e": [-1.0, 1.0, 0.0],
    "rest_mass_kg": [9.1093837e-31, 9.1093837e-31, 0.0],
    "category": ["lepton", "antilepton", "gauge boson"],
})

particles


The positron is the electron's antiparticle. This notebook only uses basic properties; it does not model production, storage, or handling of antimatter.


## 2. Photon energy

\[
E = hf = \frac{hc}{\lambda}.
\]


In [ ]:
h, c, lam = sp.symbols("h c lambda", positive=True, real=True)
E_photon = h*c/lam

display(sp.Eq(sp.Symbol("E"), E_photon))
display(sp.Eq(sp.Symbol(r"\frac{dE}{d\lambda}"), sp.diff(E_photon, lam)))


In [ ]:
h_val = 6.62607015e-34
c_val = 299792458.0
q_e = 1.602176634e-19

wavelength_nm = np.array([405, 532, 633, 1064, 1310, 1550, 1590], dtype=float)
wavelength_m = wavelength_nm * 1e-9

energy_j = h_val*c_val/wavelength_m
energy_ev = energy_j/q_e
frequency_hz = c_val/wavelength_m

photon_df = pd.DataFrame({
    "wavelength_nm": wavelength_nm,
    "frequency_THz": frequency_hz/1e12,
    "energy_eV": energy_ev,
})

photon_df


In [ ]:
plt.figure(figsize=(7,4))
plt.plot(wavelength_nm, energy_ev, marker="o")
plt.xlabel("Wavelength [nm]")
plt.ylabel("Photon energy [eV]")
plt.title("Photon energy versus wavelength")
plt.grid()
plt.show()


## 3. Laser power versus photon energy

If each photon has energy \(E_\gamma\), an idealized photon rate is

\[
\dot N = \frac{P}{E_\gamma}.
\]

This separates energy per photon from total optical power.


In [ ]:
P, Eg = sp.symbols("P E_gamma", positive=True, real=True)
display(sp.Eq(sp.Symbol(r"\dot{N}"), P/Eg))


## 4. Optical power → current → voltage

A simplified photodetector model is

\[
i = \mathcal{R}P_{\rm optical}
\]

and an idealized transimpedance relation is

\[
V_{\rm out} = -R_f i.
\]


In [ ]:
Popt, responsivity, Rf = sp.symbols(
    "P_optical responsivity R_f",
    positive=True, real=True
)

i_photo = responsivity*Popt
Vout = -Rf*i_photo

display(sp.Eq(sp.Symbol("i_photo"), i_photo))
display(sp.Eq(sp.Symbol("V_out"), Vout))


In [ ]:
Popt_num = 100e-6   # example teaching value
Rresp_num = 0.8     # A/W, example teaching value
Rf_num = 10e3       # ohm, example teaching value

i_num = Rresp_num*Popt_num
v_num = -Rf_num*i_num

print(f"Photocurrent = {i_num:.3e} A")
print(f"Output voltage = {v_num:.3f} V")


## 5. Voltage → ADC code

For an idealized \(n\)-bit ADC,

\[
N \approx \frac{V}{V_{\rm ref}}(2^n-1).
\]


In [ ]:
def ideal_adc_code(voltage, v_ref=5.0, bits=10):
    if not (0.0 <= voltage <= v_ref):
        raise ValueError("voltage must lie between 0 and v_ref")
    return int(round(voltage/v_ref * (2**bits - 1)))

for voltage in [0.5, 1.0, 2.5, 4.0]:
    n = ideal_adc_code(voltage)
    print(f"{voltage:4.1f} V -> {n:4d} -> {n:010b}")


# 6. Schrödinger equation

The 1-D time-independent Schrödinger equation is

\[
-\frac{\hbar^2}{2m}\frac{d^2\psi}{dx^2} + V(x)\psi = E\psi.
\]

After finite-difference discretization:

\[
\boxed{H\mathbf{\psi}=E\mathbf{\psi}}.
\]


In [ ]:
hbar = 1.054571817e-34
m_e = 9.1093837e-31
eV = 1.602176634e-19

N = 300
L = 10e-9

x = np.linspace(0.0, L, N)
dx = x[1] - x[0]

xi = x[1:-1]
Ni = len(xi)

D2 = (
    np.diag(-2*np.ones(Ni))
    + np.diag(np.ones(Ni-1), 1)
    + np.diag(np.ones(Ni-1), -1)
) / dx**2

V = np.zeros(Ni)

H = -(hbar**2/(2*m_e))*D2 + np.diag(V)

energies, states = np.linalg.eigh(H)
energies_ev = energies/eV

print("First five energies [eV]:")
print(energies_ev[:5])


## 7. Validate against the infinite-well formula

\[
E_n = \frac{n^2\pi^2\hbar^2}{2mL^2}.
\]


In [ ]:
n = np.arange(1, 6)

E_analytic_ev = (
    n**2*np.pi**2*hbar**2/(2*m_e*L**2)
) / eV

comparison = pd.DataFrame({
    "n": n,
    "numerical_eV": energies_ev[:5],
    "analytic_eV": E_analytic_ev,
})

comparison["relative_error"] = (
    comparison["numerical_eV"] - comparison["analytic_eV"]
) / comparison["analytic_eV"]

comparison


In [ ]:
plt.figure(figsize=(8,4))

for j in range(3):
    psi = states[:, j]
    psi = psi/np.max(np.abs(psi))
    plt.plot(xi*1e9, psi + 1.5*j, label=f"state {j+1}")

plt.xlabel("Position [nm]")
plt.ylabel("Normalized wavefunction + offset")
plt.title("First three numerical quantum states")
plt.grid()
plt.legend()
plt.show()


## 8. Quantum confinement

For the infinite well,

\[
E_1 \propto \frac{1}{L^2}.
\]

Shrinking the confinement length raises the energy scale.


In [ ]:
widths_nm = np.linspace(2, 20, 200)
widths_m = widths_nm*1e-9

E1_ev = (
    np.pi**2*hbar**2/(2*m_e*widths_m**2)
) / eV

plt.figure(figsize=(7,4))
plt.plot(widths_nm, E1_ev)
plt.xlabel("Well width [nm]")
plt.ylabel("Ground-state energy [eV]")
plt.title("Quantum confinement scaling")
plt.grid()
plt.show()


# 9. How the Schrödinger equation becomes engineering value

People generally do not pay an engineer simply for knowing

\[
\hat H\psi=E\psi.
\]

They pay for capabilities built from it.

| Physics capability | Engineering value |
|---|---|
| quantum confinement | semiconductor device modeling |
| energy levels | laser / LED analysis |
| tunneling | nanoscale-device modeling |
| eigenvalue solvers | photonic/electronic mode simulation |
| wavefunctions | materials/device simulation |
| photon detection | optical instrumentation |
| numerical PDEs | scientific software |

A realistic value chain is

\[
\boxed{
\text{equation}
\rightarrow
\text{validated simulation}
\rightarrow
\text{device prediction}
\rightarrow
\text{measurement}
\rightarrow
\text{engineering decision}
}
\]

The marketable skill is translating physics into reliable computational tools that help design, test, measure, or optimize hardware.


# 10. Practice

1. Add proton, neutron, and neutrino rows to the particle table.
2. Sweep optical power and plot photocurrent and voltage.
3. Sweep well width and tabulate \(E_1,E_2,E_3\).
4. Replace the zero potential with a finite square well.
5. Write five sentences explaining what engineering decision your model could support.

Keep experimental, simulated, and example data clearly labeled.
